# PQC-HO quickstart

This notebook performs a small deterministic check using the explicit **demo** workload profile. Demo outputs validate the software only and must not be cited as paper evidence. The full frozen experiment command is provided at the end.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
ROOT


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pqcho.experiments import SimConfig, demo_profiles, generate_jobs, simulate


## Small deterministic comparison

This uses 50 vehicles and seed 9001. It is intentionally smaller than the manuscript's 300-vehicle, 20-seed experiment.


In [ ]:
profiles = demo_profiles()
cfg = SimConfig(
    n_vehicles=50,
    arrival_window_ms=120.0,
    burst_window_ms=40.0,
    max_simulation_ms=1000.0,
)
jobs = generate_jobs(cfg, profiles, seed=9001)

rows = []
for method in ["PQCUnaware", "PQC-HO"]:
    _, metrics, _ = simulate(jobs, cfg, method, seed=9001)
    rows.append({
        "method": method,
        "mean_latency_ms": metrics["mean_latency_ms"],
        "deadline_violation_pct": 100 * metrics["deadline_violation_rate"],
        "mean_tardiness_ms": metrics["mean_tardiness_ms"],
    })
summary = pd.DataFrame(rows)
summary


In [ ]:
ax = summary.plot.bar(
    x="method", y="deadline_violation_pct", legend=False, figsize=(5, 3)
)
ax.set_ylabel("Deadline violations (%)")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## Full frozen run

From the repository root, reproduce the reported-profile experiment with:

```bash
pqcho run \
  --profiles data/pqc_profiles_reported.csv \
  --seeds 20 \
  --seed-start 7001 \
  --outdir results/final_7001_7020
```

Read `docs/REPRODUCIBILITY.md` before reporting or extending the results.
